In [1]:
import nest_asyncio
nest_asyncio.apply()  # Cho phép nest event loop trong Jupyter

import sys
sys.path.insert(0, "/home/duoc/workflow/api")  # thư mục chứa folder app/
print("Cấu hình sys.path thành công! Thư mục hiện tại:")

Cấu hình sys.path thành công! Thư mục hiện tại:


In [2]:
"""
Cell test Image RAG - Đầy đủ chức năng
Paste vào Jupyter, thay URL ảnh của bạn vào IMAGE_URL
"""

import asyncio
import sys
from pathlib import Path



PERSIST_DIR = Path("/home/duoc/workflow/api/rag_storage/image")
API_ROOT = Path("/home/duoc/workflow/api")

# ── SETUP PATH ──────────────────────────────────────────────────────
sys.path.insert(0, str(API_ROOT))

from app.rag.service import ImageRAG, ImageEmbedder
import faiss
import numpy as np
from PIL import Image
import requests
from io import BytesIO

# ════════════════════════════════════════════════════════════════════
# CELL 1: KIỂM TRA INDEX HIỆN CÓ
# ════════════════════════════════════════════════════════════════════
print("=" * 60)
print("CELL 1: KIỂM TRA INDEX HIỆN CÓ")
print("=" * 60)

fp = PERSIST_DIR / "faiss.index"
if fp.exists():
    idx = faiss.read_index(str(fp))
    print(f"✅ Index tồn tại: {idx.ntotal} vectors")
    print(f"   Dim: {idx.d}")
else:
    print(f"⚠️  Chưa có index tại {fp}")
    print("   Chạy Cell 2 để add ảnh đầu tiên")


INFO:rag:[config] device=cpu dtype=torch.float32
INFO:rag:[tokenizer] fallback regex


CELL 1: KIỂM TRA INDEX HIỆN CÓ
✅ Index tồn tại: 7 vectors
   Dim: 768


In [3]:

# ════════════════════════════════════════════════════════════════════
# CELL 2: KHỞI TẠO RAG VÀ ADD ẢNH TỪ URL
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("CELL 2: ADD ẢNH TỪ URL")
print("=" * 60)

# ── CONFIG ──────────────────────────────────────────────────────────
IMAGE_URL = (
    "https://scontent.fdad3-8.fna.fbcdn.net/v/t39.30808-6/475092591_1649998608960175_2456220910899203339_n.jpg?stp=cp6_dst-jpg_tt6&cstp=mx1536x2048&ctp=s1536x2048&_nc_cat=105&ccb=1-7&_nc_sid=833d8c&_nc_ohc=4VfCy8I54H4Q7kNvwG8Dl5c&_nc_oc=AdoKrE2DH9rrvyOHvHfQ-c_Bnlsb8JgHNVC3g8fO9figm88rrNi8mJ4KoQ131HocVx0&_nc_zt=23&_nc_ht=scontent.fdad3-8.fna&_nc_gid=yIAOOx2y-ZDjxc81imF77A&_nc_ss=7b289&oh=00_Af8TNuu2yHny38r8bAMRNLibuqCmxzTKMAS7WrOXcOrIJQ&oe=6A3F0F4C"
)

async def add_image():
    rag = ImageRAG()

    print(f"📥 Tải ảnh từ: {IMAGE_URL[:60]}...")
    resp = requests.get(IMAGE_URL, timeout=30)
    resp.raise_for_status()

    img = Image.open(BytesIO(resp.content))
    print(f"   Original size: {img.size}")

    # Resize giữ tỷ lệ, max 448px cho nhanh
    img.thumbnail((448, 448))
    if img.mode != "RGB":
        img = img.convert("RGB")
    print(f"   Resized: {img.size}")

    # test caption riêng
    if getattr(rag, "_captioner", None):
        print("\n📝 Florence caption:")
        caption = await rag.caption(img)
        print(caption)
    
    result = await rag.add(
        image=img,
        image_id="test_crab_001",
        business_id="brand_A",
        source_url=IMAGE_URL,
        tags=["seafood", "fresh", "crab"]
    )
    print("\n📦 Stored metadata:")
    print(rag._store._metas[-1])
    print(f"   Result: {result}")
    print(f"   Stats: {rag.stats()}")
    return rag

rag = await add_image()



INFO:rag:[caption] loading /home/duoc/models/florence-2-large...



CELL 2: ADD ẢNH TỪ URL
📥 Tải ảnh từ: https://scontent.fdad3-8.fna.fbcdn.net/v/t39.30808-6/4750925...
   Original size: (1536, 2048)
   Resized: (336, 448)

📝 Florence caption:


`torch_dtype` is deprecated! Use `dtype` instead!
INFO:rag:[caption] loaded


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 3: SEARCH BẰNG TEXT (TEXT-TO-IMAGE)
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("CELL 3: SEARCH BẰNG TEXT")
print("=" * 60)

async def test_text_search():
    rag = ImageRAG()

    queries = [
        "fresh crab seafood",
        "hotel room",
        "cua biển",
        "red string tied crab",
        "luxury dining",
        "street food market",
    ]

    for q in queries:
        print(q)
        results = await rag.search_by_text(q, k=3)

        if not results:
            print("   ❌ Không có kết quả")
            continue

        for r in results:
            score = r["score"]
            meta = r.get("meta", {})
            caption = meta.get("caption", "N/A")
            biz = meta.get("business_id", "N/A")
            # Hiển thị score với màu (conceptual)
            bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
            print(f"   {bar} {score:.3f} | {r['image_id']} | {caption} | biz:{biz}")

await test_text_search()


In [ ]:

# ════════════════════════════════════════════════════════════════════
# CELL 4: SEARCH BẰNG ẢNH (IMAGE-TO-IMAGE / REVERSE SEARCH)
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("CELL 4: SEARCH BẰNG ẢNH (REVERSE IMAGE SEARCH)")
print("=" * 60)

async def test_image_search():
    rag = ImageRAG()

    # Tải lại ảnh query
    print(f"📥 Query image: {IMAGE_URL[:60]}...")
    resp = requests.get(IMAGE_URL, timeout=30)
    query_img = Image.open(BytesIO(resp.content))
    query_img.thumbnail((448, 448))
    if query_img.mode != "RGB":
        query_img = query_img.convert("RGB")

    results = await rag.search(query_img, k=3)
    print(f"\n🔍 Kết quả tìm ảnh tương tự:")

    for r in results:
        score = r["score"]
        meta = r.get("meta", {})
        bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
        print(f"   {bar} {score:.3f} | {r['image_id']} | {meta.get('caption', 'N/A')}")

await test_image_search()


In [ ]:

# ════════════════════════════════════════════════════════════════════
# CELL 5: TEST EMBEDDER TRỰC TIẾP (SO SÁNH VECTOR)
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("CELL 5: SO SÁNH VECTOR TRỰC TIẾP")
print("=" * 60)

async def test_vector_comparison():
    embedder = ImageEmbedder()

    # Encode nhiều text cùng lúc để so sánh
    texts = [
        "fresh crab",
        "seafood market", 
        "hotel room",
        "red string",
        "luxury restaurant",
        "street food",
    ]

    print(f"📊 Encoding {len(texts)} text queries...")
    embs = await embedder.encode_text(texts)

    # Tải ảnh và encode
    print(f"📥 Encoding image...")
    resp = requests.get(IMAGE_URL, timeout=30)
    img = Image.open(BytesIO(resp.content))
    img.thumbnail((448, 448))
    if img.mode != "RGB":
        img = img.convert("RGB")

    img_emb = await embedder.encode([img])

    # So sánh cosine similarity
    print(f"\n📈 Similarity với ảnh cua:")
    print("-" * 50)

    img_vec = img_emb[0]
    similarities = []
    for text, text_vec in zip(texts, embs):
        # Cosine similarity = dot product (đã normalize L2)
        sim = np.dot(img_vec, text_vec)
        similarities.append((sim, text))

    # Sort giảm dần
    similarities.sort(reverse=True)

    for sim, text in similarities:
        bar = "█" * int(sim * 15) + "░" * (15 - int(sim * 15))

await test_vector_comparison()



In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 6: STATS & CLEANUP
# ════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("CELL 6: STATS")
print("=" * 60)

rag = ImageRAG()
print(f"📊 {rag.stats()}")

print("\n✅ Tất cả test hoàn thành!")

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

processor = AutoProcessor.from_pretrained(
    "/home/duoc/models/florence-2-large",
    trust_remote_code=True,
    local_files_only=True,
     attn_implementation="eager"
)

model = AutoModelForCausalLM.from_pretrained(
    "/home/duoc/models/florence-2-large",
    trust_remote_code=True,
    local_files_only=True,
     attn_implementation="eager"
)

print("OK")